# CNNs with Keras on landscape data

In this lab, we classify the photos of the landscape dataset (Intel Image Classification, 6 classes):

1. with the slides' CNN, then with a deeper CNN, which you write (optional: with batch normalization, AdamW and warmup);
2. by augmenting the data with layers;
3. with three historical architectures: LeNet, AlexNet and an Inception block;
4. by analyzing the errors on the test set;
5. by transfer learning: EfficientNetV2B0 pretrained on ImageNet, frozen then fine-tuned ("Transfer Learning" part);
6. optional: with a pretrained Vision Transformer ("Transfer Learning With Transformers" part).

Part 5 goes with the slides' "Transfer learning" chapter: stop before it if you are only at the CNN chapter. Part 6 is optional: it repeats part 5's approach with an architecture, the Vision Transformer, which the part sums up in a few lines and which the slides present later ("Transformers" chapter).

## Checking GPU Usage

Go to the `Runtime > Change runtime type` menu and check that the hardware accelerator is a GPU: without one, each training of this lab would take hours.

In [ ]:
!nvidia-smi

## Downloading the Landscape Dataset

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh landscape
!ls landscape
print("***")
!ls -l landscape/seg_train
print("***")
!ls -l landscape/seg_pred

## Imports

In [ ]:
!pip install -q keras-hub
import itertools
import pathlib

import keras
import keras_hub
import matplotlib.pyplot as plt
import numpy
import seaborn
import sklearn.metrics
from keras import layers

# Same random draws on every run: reproducible results
keras.utils.set_random_seed(0)

## Preparing the Data

As in the slides, `keras.utils.image_dataset_from_directory` reads the images of `seg_train` in batches of 32, folder by folder: an image's label is the index of its folder, in alphabetical order (0 = `buildings`, …, 5 = `street`). 20% of the images are used for validation, with a split fixed by `seed=0`.

The images stay on disk and are read at every epoch. In memory, as 4-byte floats, the 14,034 training images would take 3.8 GB at 150 × 150 and 8.5 GB at 224 × 224 (the transfer learning size): more than what Colab leaves free next to the model, out of its 12.7 GB of RAM.

The test set (`seg_test`) is read without shuffling (`shuffle=False`): the model's predictions come out in the order of `test_ds.file_paths`, the one of `test_labels`.

In [ ]:
def load_landscapes(image_size: tuple[int, int]) -> tuple:
  """Load the training, validation and test images at this size."""
  train_ds, val_ds = keras.utils.image_dataset_from_directory(
    "landscape/seg_train",
    image_size=image_size,
    validation_split=0.2,  # 20% of the images for validation
    subset="both",
    seed=0,  # the same split on every run
  )
  test_ds = keras.utils.image_dataset_from_directory(
    "landscape/seg_test", image_size=image_size, shuffle=False
  )
  return train_ds, val_ds, test_ds


train_ds, val_ds, test_ds = load_landscapes((150, 150))
label_names = train_ds.class_names
# Paths and labels of the test images, in the order of the predictions
test_paths = numpy.array(test_ds.file_paths)
test_labels = numpy.concatenate([labels for _, labels in test_ds])
print(label_names)

## Looking at the Data

In [ ]:
# A batch: 32 150 × 150 images with 3 channels, and their 32 labels
images, labels = next(iter(train_ds))
print(f"Shape of a batch of images: {images.shape}")
print(f"Shape of a batch of labels: {labels.shape}")

counts = [
  len(list((pathlib.Path("landscape/seg_train") / name).iterdir()))
  for name in label_names
]
plt.bar(label_names, counts)
plt.title("Number of images per class (training and validation)")
plt.ylabel("Number of images")
plt.xlabel("Class")
plt.show()

In [ ]:
# The pixels are floats from 0 to 255: imshow expects them as integers
images = numpy.asarray(images).astype("uint8")
_, ax = plt.subplots(5, 5, figsize=(15, 15))
for k, (i, j) in enumerate(itertools.product(range(5), repeat=2)):
  ax[i, j].imshow(images[k])
  ax[i, j].set_title(label_names[int(labels[k])])
  ax[i, j].axis("off")
plt.show()

## Creating the Model

The slides' CNN ("Defining a CNN"): two convolution + pooling blocks, then a global pooling that keeps one value per channel, and a dense output layer. `Rescaling(1 / 255)` brings the pixels from 0–255 to 0–1, the scale at which a network trained from scratch learns well.

In [ ]:
model = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    layers.Rescaling(1 / 255),  # pixels 0–255 → 0–1
    layers.Conv2D(32, 3, activation="relu"),  # 32 3 × 3 filters
    layers.MaxPooling2D(2),  # 2 × 2 windows, stride of 2
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.GlobalAveragePooling2D(),  # one value per channel
    layers.Dense(6, activation="softmax"),  # one probability per class
  ]
)
model.compile(
  optimizer=keras.optimizers.Adam(learning_rate=1e-3),
  loss="sparse_categorical_crossentropy",  # integer labels
  metrics=["accuracy"],
)
model.summary()

## Can You Explain the Different Numbers of Parameters?

Compute the number of parameters of each layer of the summary. Why do some layers have none?

### Solution

- `conv2d`: each of the 32 filters has 3 × 3 weights per input channel, over 3 channels (RGB), plus one bias: (3 × 3 × 3 + 1) × 32 = 896.
- `conv2d_1`: 64 filters, each covering the 32 channels of the previous layer: (3 × 3 × 32 + 1) × 64 = 18,496.
- `dense`: 64 inputs (one per channel, after the global pooling) and 6 outputs, plus 6 biases: 64 × 6 + 6 = 390.
- `rescaling`, the poolings and the global pooling learn nothing: they apply a fixed operation (multiplying, taking a maximum or a mean).

Total: 19,782 parameters. The global pooling keeps the head small: with `Flatten`, the dense layer would receive 36 × 36 × 64 = 82,944 values, i.e. 497,670 parameters.

## Training

Let's train this model as in the slides ("Training with Early Stopping"): at most 10 epochs, stopped after 3 epochs without a decrease of the validation loss, with a return to the best weights.

A callback can keep its state from one `fit` to the next: depending on the Keras 3 version, a reused `EarlyStopping` starts from the best validation loss of the previous training, and it would then stop the next model after 3 epochs, giving it back the weights of its first epoch. `early_stopping()` therefore creates a new one for each training.

`train(model, epochs, learning_rate)` then groups these steps (compiling, training with early stopping, curves) for the next models.

In [ ]:
def early_stopping() -> keras.callbacks.EarlyStopping:
  """A new early stopping: 3 epochs of patience, return to the best weights."""
  return keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
  )


history = model.fit(
  train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stopping()]
)


def plot_metrics(history: keras.callbacks.History) -> None:
  """Plot the accuracy and the loss, in training and in validation."""
  for metric, name in (("accuracy", "Accuracy"), ("loss", "Loss")):
    plt.plot(history.history[metric], label="Training")
    plt.plot(history.history[f"val_{metric}"], label="Validation")
    plt.title(f"Model {name.lower()}")
    plt.ylabel(name)
    plt.xlabel("Epoch")
    plt.legend()
    plt.show()


plot_metrics(history)


def train(
  model: keras.Model, epochs: int, learning_rate: float = 1e-4
) -> keras.callbacks.History:
  """Compile and train a model on the 150 × 150 images, then plot its curves."""
  model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  history = model.fit(
    train_ds, validation_data=val_ds, epochs=epochs, callbacks=[early_stopping()]
  )
  plot_metrics(history)
  return history

## Improve This Performance

The slides' CNN tops out at about 66% accuracy: two convolutions only see small patterns, too few to recognize a scene. Write a `deep_cnn()` function that returns a deeper CNN, with new weights at each call, then train it with `train`:

- convolution + pooling blocks repeated, down to feature maps a few pixels wide;
- dropout before the head's dense layers, against overfitting;
- between 10 and 20 epochs, each under a minute on Colab's GPU.

Goal: above 70% validation accuracy. The solution reaches about 86% in 15 epochs of about 45 seconds. `deep_cnn()` will be reused for data augmentation.

**Useful documentation**:

- https://keras.io/api/layers/reshaping_layers/flatten/
- https://keras.io/api/layers/regularization_layers/dropout/

In [ ]:
def deep_cnn() -> keras.Sequential: ...


# Your code here

### Additional hints

- Input: `keras.Input(shape=(150, 150, 3))` then `layers.Rescaling(1 / 255)`, as in the slides' CNN.
- A block: `layers.Conv2D(200, 3, activation="relu", padding="same", kernel_initializer="orthogonal")` then `layers.MaxPooling2D(2, padding="same")`. With `padding="same"`, the convolution keeps the size of the feature maps and only the pooling halves it, rounded up: 150 → 75 → 38 → 19 → 10 → 5 → 3 in 6 blocks. A 7th convolution, without pooling, ends on 3 × 3 feature maps.
- Head: `layers.Flatten()`, then three ReLU `Dense` layers of 200, 100 and 50 neurons, and the output `layers.Dense(6, activation="softmax")`, each preceded by `layers.Dropout(0.2)`.
- Training: `train(model, epochs=15)`, with `train`'s default learning rate (1e-4).

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def deep_cnn() -> keras.Sequential:
  blocks = []
  for _ in range(6):
    blocks += [layers.Conv2D(...), layers.MaxPooling2D(...)]
  return keras.Sequential(
    [
      keras.Input(shape=(150, 150, 3)),
      layers.Rescaling(1 / 255),
      *blocks,
      layers.Conv2D(...),
      layers.Flatten(),
      layers.Dropout(0.2),
      layers.Dense(200, activation="relu"),
      ...,
      layers.Dense(6, activation="softmax"),
    ]
  )


model = deep_cnn()
model.summary()
history = train(model, epochs=...)
```

### Solution

In [ ]:
def deep_cnn() -> keras.Sequential:
  """The solution's deep CNN, with new weights at each call."""
  blocks = []
  for _ in range(6):  # feature maps 150 → 75 → 38 → 19 → 10 → 5 → 3 pixels wide
    blocks += [
      layers.Conv2D(
        200,
        3,
        activation="relu",
        padding="same",
        kernel_initializer="orthogonal",
      ),
      layers.MaxPooling2D(2, padding="same"),
    ]
  return keras.Sequential(
    [
      keras.Input(shape=(150, 150, 3)),
      layers.Rescaling(1 / 255),
      *blocks,
      layers.Conv2D(
        200, 3, activation="relu", padding="same", kernel_initializer="orthogonal"
      ),
      layers.Flatten(),
      layers.Dropout(0.2),
      layers.Dense(200, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(100, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(50, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(6, activation="softmax"),
    ]
  )


model = deep_cnn()
model.summary()

In [ ]:
history = train(model, epochs=15)

### Optional: Batch Normalization, AdamW and Warmup

This exercise is optional, for those who finish early: the rest of the lab doesn't depend on it. It takes the deep CNN again with three techniques for training deep networks: batch normalization, the AdamW optimizer, and a learning rate that rises during a *warmup* then decays along a cosine.

Write `deep_cnn_bn()`, which returns the solution's deep CNN with a `BatchNormalization` in each block, between the convolution and its activation. Compile the model it returns with `keras.optimizers.AdamW`, whose learning rate follows a `keras.optimizers.schedules.CosineDecay` with warmup, and train it for 15 epochs. Finally, compare its learning curves to the deep CNN's (`history`, from the previous solution): which one goes above 70% validation accuracy first? Which one ends highest? Does the gap between training and validation change?

**Useful documentation**:

- https://keras.io/api/layers/normalization_layers/batch_normalization/
- https://keras.io/api/optimizers/adamw/
- https://keras.io/api/optimizers/learning_rate_schedules/cosine_decay/

In [ ]:
# Your code here

#### Additional hints

- A block's convolution, without activation or bias: `layers.Conv2D(200, 3, padding="same", kernel_initializer="orthogonal", use_bias=False)`, then `layers.BatchNormalization()` and `layers.Activation("relu")`. BatchNorm recenters the activations: a bias before it would be useless. The solution returns them from a function, which serves the 6 blocks (each followed by `layers.MaxPooling2D(2, padding="same")`) and the 7th convolution. The head doesn't change.
- Schedule, counted in batches: `steps = epochs * len(train_ds)` (`len` gives the number of batches in an epoch) and `warmup = steps // 20`, 5% of the steps. `keras.optimizers.schedules.CosineDecay(initial_learning_rate=0.0, warmup_target=1e-3, warmup_steps=warmup, decay_steps=steps - warmup)` rises in a straight line from 0 to 1e-3 during the warmup, then goes back down along a cosine to 0. This peak is ten times `train`'s 1e-4: BatchNorm usually tolerates a higher learning rate. It isn't tuned: a *learning rate range test* would choose it.
- Optimizer: `keras.optimizers.AdamW(schedule, weight_decay=0.01)`, then `optimizer.exclude_from_weight_decay(var_names=["bias", "gamma", "beta"])`: neither the biases nor the BatchNorm parameters are regularized.
- `train` compiles with Adam at a fixed rate: compile yourself, with the same loss and the same metric, then `model_bn.fit(train_ds, validation_data=val_ds, epochs=epochs)`, without early stopping: the schedule plans for every epoch, and the rate only reaches 0 at the last one.
- Curves: `plot_metrics(history_bn)`, then, to compare, `history.history["val_accuracy"]` and `history_bn.history["val_accuracy"]` on the same plot (two `plt.plot(..., label=...)`), and likewise for `"val_loss"`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def deep_cnn_bn() -> keras.Sequential:
  blocks = []
  for _ in range(6):
    blocks += [
      layers.Conv2D(...),
      layers.BatchNormalization(),
      ...,
      layers.MaxPooling2D(2, padding="same"),
    ]
  return keras.Sequential(
    [
      keras.Input(shape=(150, 150, 3)),
      layers.Rescaling(1 / 255),
      *blocks,
      ...,
      layers.Flatten(),
      ...,
    ]
  )


model_bn = deep_cnn_bn()
epochs = 15
steps = epochs * len(train_ds)
warmup = steps // 20
schedule = keras.optimizers.schedules.CosineDecay(...)
optimizer = keras.optimizers.AdamW(schedule, weight_decay=0.01)
optimizer.exclude_from_weight_decay(var_names=["bias", "gamma", "beta"])
model_bn.compile(optimizer=optimizer, loss=..., metrics=...)
history_bn = model_bn.fit(...)
plot_metrics(history_bn)
for metric, name in (("val_accuracy", "Accuracy"), ("val_loss", "Loss")):
  plt.plot(history.history[metric], label="Deep CNN: Adam at 1e-4")
  plt.plot(...)
  ...
  plt.show()
```

#### Solution

In [ ]:
def conv_bn_layers() -> list:
  """A convolution without bias or activation, its BatchNorm, then the ReLU."""
  return [
    layers.Conv2D(
      200, 3, padding="same", kernel_initializer="orthogonal", use_bias=False
    ),
    layers.BatchNormalization(),  # BatchNorm recenters: no bias needed
    layers.Activation("relu"),
  ]


def deep_cnn_bn() -> keras.Sequential:
  """The solution's deep CNN, with a BatchNorm in each block."""
  blocks = []
  for _ in range(6):  # feature maps 150 → 75 → 38 → 19 → 10 → 5 → 3 pixels wide
    blocks += [*conv_bn_layers(), layers.MaxPooling2D(2, padding="same")]
  return keras.Sequential(
    [
      keras.Input(shape=(150, 150, 3)),
      layers.Rescaling(1 / 255),
      *blocks,
      *conv_bn_layers(),
      layers.Flatten(),
      layers.Dropout(0.2),
      layers.Dense(200, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(100, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(50, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(6, activation="softmax"),
    ]
  )


model_bn = deep_cnn_bn()
model_bn.summary()

In [ ]:
epochs = 15
steps = epochs * len(train_ds)  # in batches
warmup = steps // 20  # 5% of the steps
schedule = keras.optimizers.schedules.CosineDecay(
  initial_learning_rate=0.0,
  warmup_target=1e-3,  # peak, reached at the end of the warmup
  warmup_steps=warmup,
  decay_steps=steps - warmup,
)
optimizer = keras.optimizers.AdamW(schedule, weight_decay=0.01)
# Neither the biases nor the BatchNorm parameters (gamma, beta) are regularized
optimizer.exclude_from_weight_decay(var_names=["bias", "gamma", "beta"])
model_bn.compile(
  optimizer=optimizer,
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
# No early stopping: the schedule plans for every epoch
history_bn = model_bn.fit(train_ds, validation_data=val_ds, epochs=epochs)
plot_metrics(history_bn)

# Validation of both deep CNNs, epoch by epoch
for metric, name in (("val_accuracy", "Accuracy"), ("val_loss", "Loss")):
  plt.plot(history.history[metric], label="Deep CNN: Adam at 1e-4")
  plt.plot(history_bn.history[metric], label="BatchNorm, AdamW, warmup + cosine")
  plt.title(f"Validation {name.lower()}")
  plt.ylabel(name)
  plt.xlabel("Epoch")
  plt.legend()
  plt.show()

## Data Augmentation

When the training accuracy keeps rising while the validation accuracy stalls, the model overfits. Data augmentation (slide "Augmenting the Data with Layers") shows it variants of the training images at every epoch. Its layers only act during `fit`: `evaluate` and `predict` see the images untouched.

Choose two or three transformations that keep a landscape's class, with their amplitude: which ones don't keep it (a vertical flip? a 90° rotation? a zoom that only keeps a tenth of the image?)? Put their layers in front of the same architecture, with new weights (`deep_cnn()`), train for 15 epochs, and compare the gap between the training and validation curves to the one of the deep CNN without augmentation.

**Useful documentation**:

- https://keras.io/api/layers/preprocessing_layers/image_augmentation/

In [ ]:
# Your code here

### Additional hints

- A Keras model is used like a layer: `keras.Sequential` accepts `deep_cnn()` in its list, after `keras.Input(shape=(150, 150, 3))` and the augmentation layers.
- Layers that keep a landscape's class, and their parameters:
  - `layers.RandomFlip("horizontal")`: left-right flip, half of the time;
  - `layers.RandomZoom((-0.3, 0))`: zoom in by up to 30% (negative values zoom in);
  - `layers.RandomRotation(0.03)`: rotation of at most 3% of a turn, about 10°, either way;
  - `layers.RandomTranslation(0.1, 0.1)`: shift of at most 10% of the height and width;
  - `layers.RandomContrast(0.2)`: contrast multiplied by a factor between 0.8 and 1.2.
- To avoid: `layers.RandomFlip("vertical")` (the sky would end up at the bottom), large rotations, a zoom so strong it only keeps a bit of sky or wall, a hue change that would make the sea green or a glacier yellow.
- These layers work on the pixels from 0 to 255, before `deep_cnn()`'s `Rescaling`.
- Training: `train(model, epochs=15)`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
model = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    ...,
    ...,
    deep_cnn(),
  ]
)
model.summary()
history = train(model, epochs=15)
```

### Solution

In [ ]:
model = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    layers.RandomFlip("horizontal"),  # mirror image, half of the time
    layers.RandomZoom((-0.3, 0)),  # zoom in, by up to 30%
    deep_cnn(),
  ]
)
model.summary()
history = train(model, epochs=15)

One solution among others: the horizontal flip and the zoom in. Each epoch sees different variants of the images: the training accuracy rises more slowly and the gap with validation shrinks. The model has to learn features that withstand framing and orientation, instead of memorizing the training images. The test evaluation and the error analysis are about this model, `model`.

## Historical Architectures

Three historical architectures, to train and then compare with the deep CNN: LeNet and AlexNet, seen in the CNN chapter, and an Inception block (GoogLeNet, 2014), presented below. Their code is given. Each one has its own variable name, and `model` stays the augmented deep CNN.

### LeNet

[LeNet-5](https://en.wikipedia.org/wiki/LeNet) (1998), designed for 32 × 32 handwritten digits: 5 × 5 convolutions, sigmoid activations and average poolings.

In [ ]:
le_net = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    layers.Rescaling(1 / 255),
    layers.Conv2D(6, 5, padding="same", activation="sigmoid"),
    layers.AveragePooling2D(2),
    layers.Conv2D(16, 5, activation="sigmoid"),
    layers.AveragePooling2D(2),
    layers.Flatten(),
    layers.Dense(120, activation="sigmoid"),
    layers.Dense(84, activation="sigmoid"),
    layers.Dense(6, activation="softmax"),
  ],
  name="le_net",
)
le_net.summary()
le_net_history = train(le_net, epochs=15)

### AlexNet

[AlexNet](https://en.wikipedia.org/wiki/AlexNet) (2012), ImageNet's winner: a first 11 × 11 convolution with a stride of 4, ReLUs, dropout in a head of two layers of 4,096 neurons.

In [ ]:
alex_net = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    layers.Rescaling(1 / 255),
    layers.Conv2D(96, 11, strides=4, activation="relu"),
    layers.MaxPooling2D(3, strides=2),
    layers.Conv2D(256, 5, padding="same", activation="relu"),
    layers.MaxPooling2D(3, strides=2),
    layers.Conv2D(384, 3, padding="same", activation="relu"),
    layers.Conv2D(384, 3, padding="same", activation="relu"),
    layers.Conv2D(256, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(3, strides=2),
    layers.Flatten(),
    layers.Dense(4096, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(4096, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(6, activation="softmax"),
  ],
  name="alex_net",
)
alex_net.summary()
alex_net_history = train(alex_net, epochs=15)

### Inception Block

An [Inception](https://en.wikipedia.org/wiki/Inception_(deep_learning_architecture)) block (GoogLeNet, 2014): four parallel branches (1 × 1, 3 × 3, 5 × 5 convolutions and pooling), concatenated along the channel axis. A full network stacks several of them. As in GoogLeNet, the block comes here after two convolution + pooling blocks: its 1 × 1 convolutions then bring the 128 incoming channels down to 64, which makes the 3 × 3 and 5 × 5 convolutions after them cheaper. A global pooling ends the model, as in the slides' CNN: 249,606 parameters in all. The parallel branches call for the [Keras functional API](https://keras.io/guides/functional_api/), where each layer is applied to a tensor.

In [ ]:
def inception_block(inputs: keras.KerasTensor, filters: int = 64) -> keras.KerasTensor:
  # 1 × 1 conv
  branch_1 = layers.Conv2D(filters, 1, activation="relu")(inputs)

  # 1 × 1 conv then 3 × 3 conv
  branch_3 = layers.Conv2D(filters, 1, activation="relu")(inputs)
  branch_3 = layers.Conv2D(filters, 3, padding="same", activation="relu")(branch_3)

  # 1 × 1 conv then 5 × 5 conv
  branch_5 = layers.Conv2D(filters, 1, activation="relu")(inputs)
  branch_5 = layers.Conv2D(filters, 5, padding="same", activation="relu")(branch_5)

  # 3 × 3 max pooling with a stride of 1 then 1 × 1 conv
  branch_pool = layers.MaxPooling2D(3, strides=1, padding="same")(inputs)
  branch_pool = layers.Conv2D(filters, 1, activation="relu")(branch_pool)

  # concatenation along the channel axis: 4 × 64 = 256 channels
  return layers.Concatenate()([branch_1, branch_3, branch_5, branch_pool])


inputs = keras.Input(shape=(150, 150, 3))
x = layers.Rescaling(1 / 255)(inputs)
# Two convolution + pooling blocks: 36 × 36 maps with 128 channels
x = layers.Conv2D(64, 3, activation="relu")(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(128, 3, activation="relu")(x)
x = layers.MaxPooling2D(2)(x)
x = inception_block(x)  # 1 × 1 convolutions: from 128 down to 64 channels
x = layers.GlobalAveragePooling2D()(x)  # one value per channel
outputs = layers.Dense(6, activation="softmax")(x)
inception = keras.Model(inputs, outputs, name="inception")
inception.summary()
inception_history = train(inception, epochs=10)

## Evaluating the Performance on the Test Set

The `seg_test` folder holds 3,000 images that the model has seen neither in training nor for early stopping: its accuracy on them is an honest estimate of the one it will have on new photos. `test_accuracies` keeps the test accuracy of each approach, to compare them at the end of the lab.

In [ ]:
loss, accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {accuracy:.3f}")
test_accuracies = {"Deep CNN, augmented data": accuracy}

## Error Analysis

We display the confusion matrix, then misclassified images.

In [ ]:
def analyze_preds(preds: numpy.ndarray, labels: numpy.ndarray) -> None:
  confusion_matrix = sklearn.metrics.confusion_matrix(labels, preds)
  seaborn.heatmap(
    confusion_matrix,
    annot=True,
    fmt="d",
    cmap="rocket_r",
    xticklabels=label_names,
    yticklabels=label_names,
  )
  plt.title("Confusion matrix")
  plt.xlabel("Predicted class")
  plt.ylabel("True class")
  plt.show()

  seaborn.countplot(x=[label_names[pred] for pred in preds])
  plt.title("Counts of the predicted classes")
  plt.ylabel("Count")
  plt.xlabel("Class")
  plt.show()


test_pred = model.predict(test_ds, verbose=0).argmax(axis=-1)
analyze_preds(test_pred, test_labels)

In [ ]:
def plot_mistakes(predicted_class: str, true_class: str) -> None:
  """Display up to 25 test images of true_class predicted as predicted_class."""
  print(f"Prediction: {predicted_class}, true class: {true_class}")
  mistakes = test_paths[
    (test_pred == label_names.index(predicted_class))
    & (test_labels == label_names.index(true_class))
  ]
  mistakes = numpy.random.choice(mistakes, size=min(len(mistakes), 25), replace=False)

  _, ax = plt.subplots(5, 5, figsize=(15, 15))
  grid_indexes = list(itertools.product(range(5), repeat=2))
  for i, j in grid_indexes:
    ax[i, j].axis("off")
  for path, (i, j) in zip(mistakes, grid_indexes):
    ax[i, j].imshow(keras.utils.load_img(path))
  plt.show()

In [ ]:
# Images predicted as glacier while they are labeled mountain
plot_mistakes("glacier", "mountain")

In [ ]:
# Images predicted as glacier while they are labeled sea
plot_mistakes("glacier", "sea")

In [ ]:
# Images predicted as buildings while they are labeled street
plot_mistakes("buildings", "street")

## Transfer Learning

We reuse a network pretrained on ImageNet, EfficientNetV2B0 (`keras.applications`), as a feature extractor: its weights are frozen, and we only train a new classification head for our 6 classes (slides of the "Transfer learning" chapter).

The pretrained weights expect the images as they were during their training:

- **Size**: 224 × 224, the size of the ImageNet images the weights learned on. We reload the landscapes at this size.
- **Pixels from 0 to 255**: EfficientNetV2 normalizes them itself, in its first layer. The model therefore doesn't start with `Rescaling(1 / 255)`, unlike the CNNs trained from scratch: it would only see nearly black images.

`base.trainable = False` excludes EfficientNetV2's weights from gradient descent. Its `BatchNormalization` layers then become non-trainable: they normalize with ImageNet's statistics, without updating them on our batches of landscapes.

In [ ]:
TRANSFER_SIZE = (224, 224)  # training size of the ImageNet weights
# Same images, in the same order: test_paths and test_labels remain valid
train_ds, val_ds, test_ds = load_landscapes(TRANSFER_SIZE)

In [ ]:
base = keras.applications.EfficientNetV2B0(
  include_top=False,  # without the ImageNet head (1000 classes)
  weights="imagenet",
  pooling="avg",  # GlobalAveragePooling2D: a vector of 1280 values per image
)
base.trainable = False  # frozen weights, BatchNorm in inference mode

inputs = keras.Input(shape=(224, 224, 3))  # pixels 0–255
x = base(inputs)
outputs = layers.Dense(6, activation="softmax")(x)  # 6 landscapes
model = keras.Model(inputs, outputs)
model.compile(
  optimizer=keras.optimizers.Adam(1e-3),
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
model.summary()

In [ ]:
history = model.fit(
  train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stopping()]
)
plot_metrics(history)

In [ ]:
loss, accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {accuracy:.3f}")
test_accuracies["Frozen EfficientNetV2B0"] = accuracy

test_pred = model.predict(test_ds, verbose=0).argmax(axis=-1)
analyze_preds(test_pred, test_labels)
plot_mistakes("glacier", "mountain")
plot_mistakes("glacier", "sea")
plot_mistakes("buildings", "street")

### Questions

1. How many parameters does this model train, and how many does it freeze? (see its summary)
2. Compare its test accuracy with the one of the deep CNN trained from scratch, and the time of their trainings.
3. Do the remaining errors (confusion matrix, misclassified images) look like the CNN's?

#### Solution

1. Only the head is trained: 1,280 × 6 + 6 = 7,686 parameters, against 5,919,312 frozen in EfficientNetV2B0.
2. The pretrained model beats the deep CNN, about 93% against 86% on the test set, from the first epochs and in a few seconds per epoch: only the head learns. The features EfficientNetV2 learned on over a million ImageNet images transfer well to our 14,000 landscapes.
3. The remaining confusions are mostly between really close classes (glacier and mountain, buildings and street); many of these images are ambiguous, or even mislabeled.

### Fine-Tuning: Unfreezing the Backbone

Once the head is trained, we can unfreeze EfficientNetV2 and train the whole network for a few more epochs, with a learning rate 100 times lower: its weights are already good, we only want to adjust them to our landscapes.

The next cell first keeps the weights of the model with its trained head (`model.save_weights`), to start again from the same point for the partial unfreezing, then does the slides' fine-tuning ("Fine-Tuning: Unfreezing the Backbone"): `base` entirely unfrozen, except its BatchNorm layers, which keep ImageNet's statistics, and Adam at 1e-5 for 3 epochs. Keras applies a single learning rate to the whole model: the head, already trained, also moves at 1e-5.

In [ ]:
# Weights after the head's training, to restart from them for the partial unfreezing
model.save_weights("head.weights.h5")

base.trainable = True
# Frozen BatchNorm layers: ImageNet's statistics kept
for layer in base.layers:
  if isinstance(layer, layers.BatchNormalization):
    layer.trainable = False

# To redo after any change of trainable
model.compile(
  optimizer=keras.optimizers.Adam(1e-5),
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
model.summary()
history = model.fit(train_ds, validation_data=val_ds, epochs=3)
plot_metrics(history)
loss, accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {accuracy:.3f}")
test_accuracies["EfficientNetV2B0, full unfreezing"] = accuracy

The accuracy gains about one more point, from 93 to 94% on the test set: the backbone adapts to our landscapes. A learning rate too high would on the contrary destroy what it learned on ImageNet.

### Partial Unfreezing

With little data, one often unfreezes only the last layers of the backbone (the slide "Fine-Tuning: Unfreezing the Backbone" suggests it in a comment): the first ones, which detect edges and textures, serve as they are from one image dataset to another.

Start again from the weights after the head's training, then unfreeze only the last 40 layers of `base`, BatchNorm layers still frozen. Recompile with Adam at 1e-5, train for 3 epochs and add the test accuracy to `test_accuracies`. Compare with the full unfreezing: number of trainable parameters, duration of an epoch, validation and test accuracy.

**Useful documentation**:

- https://keras.io/api/models/model_saving_apis/weights_saving_and_loading/

In [ ]:
# Your code here

#### Additional hints

- `model.load_weights("head.weights.h5")` gives the model back its weights from before the full unfreezing, head included.
- `base.trainable = True` only reaches its layers when it changes the value: after the full unfreezing, it is already `True` and would do nothing. So set each layer: `trainable = False` on `base.layers[:-40]`, all but the last 40 (the end of the last block, `block6f`, and the final convolution), and, on `base.layers[-40:]`, `True` except for the `layers.BatchNormalization` layers.
- The last 40 layers start at `block6f_se_expand`, in the middle of the `block6f` block. The keras.io example *Image classification via fine-tuning with EfficientNet* unfreezes whole blocks instead, because of each block's residual connection from its input to its output: a variant to try, the test `layer.name.startswith(("block6", "top"))`.
- `model.compile(...)`, as for the full unfreezing: without it, the change of `trainable` is ignored. `model.summary()` then shows the number of trainable parameters.
- `model.fit(train_ds, validation_data=val_ds, epochs=3)`, then `plot_metrics` and `model.evaluate(test_ds, verbose=0)`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
model.load_weights(...)

for layer in base.layers[...]:
  layer.trainable = ...
for layer in base.layers[...]:
  layer.trainable = not isinstance(layer, ...)

model.compile(...)
model.summary()
history = model.fit(train_ds, validation_data=val_ds, epochs=...)
plot_metrics(history)
loss, accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {accuracy:.3f}")
test_accuracies["EfficientNetV2B0, partial unfreezing"] = accuracy
```

#### Solution

In [ ]:
model.load_weights("head.weights.h5")  # the trained head, ImageNet's backbone

# base.trainable is already True: set each layer
for layer in base.layers[:-40]:  # every layer but the last 40
  layer.trainable = False
for layer in base.layers[-40:]:  # the last 40, BatchNorm layers frozen
  layer.trainable = not isinstance(layer, layers.BatchNormalization)

model.compile(
  optimizer=keras.optimizers.Adam(1e-5),
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
model.summary()
history = model.fit(train_ds, validation_data=val_ds, epochs=3)
plot_metrics(history)
loss, accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {accuracy:.3f}")
test_accuracies["EfficientNetV2B0, partial unfreezing"] = accuracy

- The partial unfreezing trains 1,660,134 parameters (1,652,448 in the last 40 layers of `base`, 7,686 in the head), against 5,805,782 for the full unfreezing. The gradient only goes through the end of the network: each epoch is shorter.
- The partial unfreezing may be enough here, which your test accuracies will tell: our photos are close to ImageNet's, and EfficientNetV2's first layers have little to relearn on them. On images further from them (X-rays, satellite views), the full unfreezing has more to gain.
- The rest of the lab uses `model`, fine-tuned with the partial unfreezing.

## Predicting in "Real" Conditions

The `seg_pred` folder holds unlabeled images: we can't measure the accuracy on them. We can however display photos and the probabilities the model gives to each class.

In [ ]:
pred_paths = sorted(pathlib.Path("landscape/seg_pred").rglob("*.jpg"))
print(f"{len(pred_paths)} unlabeled images")

In [ ]:
# 25 random images, without replacement, loaded at the model's size
indexes = numpy.random.choice(len(pred_paths), size=25, replace=False)
pred_images = numpy.stack(
  [
    keras.utils.img_to_array(
      keras.utils.load_img(pred_paths[index], target_size=TRANSFER_SIZE)
    )
    for index in indexes
  ]
)
probabilities = model.predict(pred_images, verbose=0)

# Each image, then the distribution of its predictions on the line below
_, ax = plt.subplots(10, 5, figsize=(30, 45))
for k, (i, j) in enumerate(itertools.product(range(5), repeat=2)):
  predicted_class = label_names[probabilities[k].argmax()]
  ax[i * 2, j].imshow(pred_images[k].astype("uint8"))
  ax[i * 2, j].set_title(f"{pred_paths[indexes[k]].name} ({predicted_class})")
  ax[i * 2, j].axis("off")
  ax[i * 2 + 1, j].bar(label_names, probabilities[k])
plt.show()

## Accuracy as a Function of the Maximum Probability

Does the accuracy vary with the maximum probability the model returns? If so, we can keep only the confident predictions, and hand the others to a human.

In [ ]:
test_probabilities = model.predict(test_ds, verbose=0)

In [ ]:
def threshold_accuracy(
  probabilities: numpy.ndarray, labels: numpy.ndarray, threshold: float
) -> tuple[float, float, float, float]:
  predictions = probabilities.argmax(axis=-1)
  mask_above = probabilities.max(axis=-1) > threshold
  mask_below = ~mask_above

  n_above = mask_above.sum()
  n_below = mask_below.sum()

  if n_above:
    above = (predictions[mask_above] == labels[mask_above]).sum() / n_above
  else:
    above = 1.0
  if n_below:
    below = (predictions[mask_below] == labels[mask_below]).sum() / n_below
  else:
    below = 1.0
  return above, below, n_above / labels.shape[0], n_below / labels.shape[0]


accuracy_above, accuracy_below, ratio_above, ratio_below = threshold_accuracy(
  test_probabilities, test_labels, 0.99
)
print(
  "Accuracy for the model's predictions with a probability above "
  f"the threshold ({ratio_above * 100:.2f}% of the data): "
  f"{accuracy_above:.2f}"
)
print(
  "Accuracy for the model's predictions with a probability below "
  f"the threshold ({ratio_below * 100:.2f}% of the data): "
  f"{accuracy_below:.2f}"
)

xs = numpy.arange(101) / 100
accuracies = []
ratios_above = []
for x in xs:
  accuracy, _, ratio_above, _ = threshold_accuracy(test_probabilities, test_labels, x)
  accuracies.append(accuracy)
  ratios_above.append(ratio_above)
plt.plot(xs, accuracies, label="Accuracy")
plt.plot(xs, ratios_above, label="Share of predictions kept")
plt.xlabel("Prediction threshold")
plt.title("Accuracy and share of predictions kept by threshold")
plt.legend()
plt.show()

## Transfer Learning With Transformers

Optional part: the slides present the Vision Transformer later ("Transformers" chapter); what you need to know about it here fits in the next paragraph.

The same approach with a Vision Transformer (ViT) pretrained on ImageNet-21k (14 million images), loaded with KerasHub. The ViT cuts a 224 × 224 image into 196 squares of 16 × 16 pixels, and adds in front of them a classification token, `[CLS]`. This token's last state summarizes the image: the classification head is plugged into it.

Build `vit_model`:

- the `"vit_base_patch16_224_imagenet21k"` preset, loaded twice: `keras_hub.layers.ViTImageConverter`, its preprocessing (resizing to 224 × 224, bringing the pixels from 0–255 to [-1, 1]), and `keras_hub.models.ViTBackbone`, the network, to freeze;
- with the functional API: a 224 × 224 × 3 input, the preprocessing, the ViT, the `[CLS]` token, then a dense output layer with 6 classes;
- compiled like the EfficientNetV2 model.

**Useful documentation**:

- https://keras.io/keras_hub/api/models/vit/vit_backbone/
- https://keras.io/keras_hub/api/models/vit/vit_image_converter/

In [ ]:
# Your code here

### Additional hints

- Loading: `keras_hub.layers.ViTImageConverter.from_preset(PRESET)` and `keras_hub.models.ViTBackbone.from_preset(PRESET)` (330 MB to download).
- The converter is used like a layer, right after `keras.Input(shape=(224, 224, 3))`.
- The ViT returns a vector of 768 values per token, of shape `(None, 197, 768)`: the `[CLS]` token is the first one, `x[:, 0]`.
- `vit.trainable = False` is enough: the ViT normalizes with `LayerNormalization` layers, which have no statistics to freeze, unlike BatchNorm.
- Compiling: `keras.optimizers.Adam(1e-3)`, `loss="sparse_categorical_crossentropy"`, `metrics=["accuracy"]`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
PRESET = "vit_base_patch16_224_imagenet21k"
converter = keras_hub.layers.ViTImageConverter.from_preset(PRESET)
vit = ...
vit.trainable = ...

inputs = keras.Input(shape=(224, 224, 3))
x = converter(inputs)
x = ...
x = x[...]
outputs = ...
vit_model = keras.Model(inputs, outputs, name="vit_model")
vit_model.compile(...)
vit_model.summary()
```

### Solution

In [ ]:
PRESET = "vit_base_patch16_224_imagenet21k"
converter = keras_hub.layers.ViTImageConverter.from_preset(PRESET)
vit = keras_hub.models.ViTBackbone.from_preset(PRESET)
vit.trainable = False

inputs = keras.Input(shape=(224, 224, 3))  # pixels 0–255
x = converter(inputs)  # 224 × 224, pixels in [-1, 1]
x = vit(x)  # (None, 197, 768): one vector per token
x = x[:, 0]  # the [CLS] token, which summarizes the image
outputs = layers.Dense(6, activation="softmax")(x)
vit_model = keras.Model(inputs, outputs, name="vit_model")
vit_model.compile(
  optimizer=keras.optimizers.Adam(1e-3),
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
vit_model.summary()

### Comparing the Three Approaches

Train `vit_model` for 5 epochs (about a minute each on Colab's GPU), evaluate it on the test set, then compare the three approaches: CNN trained from scratch, pretrained EfficientNetV2B0, pretrained ViT. What does transfer learning bring, and at what cost?

In [ ]:
history = vit_model.fit(
  train_ds, validation_data=val_ds, epochs=5, callbacks=[early_stopping()]
)
plot_metrics(history)

In [ ]:
loss, accuracy = vit_model.evaluate(test_ds, verbose=0)
test_accuracies["Frozen ViT"] = accuracy

test_pred = vit_model.predict(test_ds, verbose=0).argmax(axis=-1)
analyze_preds(test_pred, test_labels)

for name, accuracy in test_accuracies.items():
  print(f"{name}: {accuracy:.3f}")

#### Solution

- Both pretrained models clearly beat the CNN trained from scratch (93 to 94% against about 86%), for a fraction of its training time: that is the whole point of transfer learning when there is little data (14,000 images here, against over a million for ImageNet and 14 million for ImageNet-21k).
- Between EfficientNetV2B0 and the ViT, the gap is small, but the ViT is much bigger: 86 million parameters against 5.9 million, and each prediction costs accordingly. Fine-tuning the backbone brings one more point.
- The remaining confusions are the same for the three models: glacier and mountain, buildings and street.